# Parte 1 — Ingestão e integração

**Exercício Prático — Análise Integrada de Acidentes Rodoviários Federais (2023 a 2025)**

| | |
|---|---|
| **Fonte** | Dados Abertos da PRF — `datatran2023.csv`, `datatran2024.csv`, `datatran2025.csv` |
| **Grão** | 1 linha = 1 ocorrência de acidente |
| **Saída desta parte** | `acidentes_prf_historico` (DataFrame em memória) |

> Notebook executado com *Restart & Run All* sem erro. Caminhos relativos; nenhum CSV original é sobrescrito.


## 0. Configuração do ambiente

Resolve a raiz do projeto por `Path` (funciona aberto da raiz do repositório ou de
`notebooks/`) e define as constantes usadas nas partes.


In [1]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 120)


def localizar_raiz() -> Path:
    """Sobe a partir do diretório atual até achar a pasta de dados brutos."""
    for candidato in [Path.cwd(), *Path.cwd().parents]:
        if (candidato / "dados_brutos").is_dir():
            return candidato
    raise FileNotFoundError(
        "Não encontrei a pasta 'dados_brutos/'. "
        "Abra o notebook a partir da raiz do projeto ou de 'notebooks/'."
    )


RAIZ = localizar_raiz()
DIR_BRUTOS = RAIZ / "dados_brutos"
DIR_TRATADOS = RAIZ / "dados_tratados"
DIR_RESULTADOS = RAIZ / "resultados"

for pasta in (DIR_TRATADOS, DIR_RESULTADOS):
    pasta.mkdir(parents=True, exist_ok=True)

SEPARADOR = ";"
ENCODING_SAIDA = "utf-8-sig"
ENCODINGS_TENTATIVA = ("utf-8", "latin1", "utf-8-sig", "cp1252")
ANOS = (2023, 2024, 2025)

print(f"Raiz do projeto : {RAIZ}")
print(f"Dados brutos    : {DIR_BRUTOS}")
print(f"Dados tratados  : {DIR_TRATADOS}")
print(f"Resultados      : {DIR_RESULTADOS}")


Raiz do projeto : /home/user/PRF-pandas-e-Plotly---2023-2025
Dados brutos    : /home/user/PRF-pandas-e-Plotly---2023-2025/dados_brutos
Dados tratados  : /home/user/PRF-pandas-e-Plotly---2023-2025/dados_tratados
Resultados      : /home/user/PRF-pandas-e-Plotly---2023-2025/resultados


## 1.1 Leitura dos três CSVs com `ler_csv_prf` (fallback de encoding)

**Pergunta:** os três arquivos podem ser lidos sem perda de caracteres?

A função tenta cada encoding da lista até um funcionar. Os arquivos da PRF são **latin1**
(sem BOM); `utf-8` falha, então o fallback é realmente exercitado — não é código decorativo.

O `id` é lido como texto de propósito: em 2024 existe uma linha com `id = "6e+05"`
(notação científica gerada por planilha). Lido como número, o pandas converteria a coluna
inteira para `float` e essa linha perderia o identificador.


In [2]:
def ler_csv_prf(caminho: Path, sep: str = SEPARADOR) -> pd.DataFrame:
    """Lê um CSV da PRF tentando vários encodings em sequência.

    Diferença em relação à versão do Módulo 04: um encoding que falha não
    interrompe a função (não há 'raise' dentro do laço); a próxima tentativa
    é executada e o erro só é levantado se todas falharem.
    """
    ultimo_erro: Exception | None = None
    for encoding in ENCODINGS_TENTATIVA:
        try:
            df = pd.read_csv(
                caminho, sep=sep, encoding=encoding, low_memory=False,
                dtype={"id": "string"},
            )
        except (UnicodeDecodeError, UnicodeError) as erro:
            ultimo_erro = erro
            continue
        print(f"{caminho.name}: lido com encoding={encoding!r} "
              f"({len(df):,} linhas x {df.shape[1]} colunas)")
        return df
    raise RuntimeError(
        f"Nenhum encoding de {ENCODINGS_TENTATIVA} funcionou para {caminho}"
    ) from ultimo_erro

dados_por_ano = {ano: ler_csv_prf(DIR_BRUTOS / f"datatran{ano}.csv") for ano in ANOS}
linhas_por_arquivo = {ano: len(df) for ano, df in dados_por_ano.items()}
linhas_por_arquivo


datatran2023.csv: lido com encoding='latin1' (67,766 linhas x 30 colunas)


datatran2024.csv: lido com encoding='latin1' (73,202 linhas x 30 colunas)


datatran2025.csv: lido com encoding='latin1' (72,529 linhas x 30 colunas)


{2023: 67766, 2024: 73202, 2025: 72529}

## 1.2 Padronização dos nomes e conferência de schema entre os anos

**Pergunta:** os três anos têm as mesmas colunas?

A comparação é feita **antes** da concatenação: se um ano trouxesse coluna a mais ou a menos,
ela aparece aqui em vez de virar `NaN` silencioso.


In [3]:
def normalizar_nome_coluna(nome: str) -> str:
    """minúsculas, sem acento, separador único em '_'."""
    nome = str(nome).strip().lower()
    nome = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode("utf-8")
    for antigo, novo in ((" ", "_"), ("-", "_"), ("/", "_")):
        nome = nome.replace(antigo, novo)
    while "__" in nome:
        nome = nome.replace("__", "_")
    return nome.strip("_")

for df in dados_por_ano.values():
    df.columns = [normalizar_nome_coluna(c) for c in df.columns]

colunas_por_ano = {ano: set(df.columns) for ano, df in dados_por_ano.items()}
todas = set().union(*colunas_por_ano.values())
comuns = set.intersection(*colunas_por_ano.values())

print("Colunas por ano: " + " | ".join(f"{a}: {len(c)}" for a, c in colunas_por_ano.items()))
print(f"Comuns aos três anos: {len(comuns)}")
print(f"Só em alguns anos    : {sorted(todas - comuns) or 'nenhuma'}")

assert len(comuns) == len(todas), "Há colunas que não existem em todos os anos."
assert len(comuns) == 30, f"Esperava 30 colunas, encontrei {len(comuns)}."
print("\nOK — os três anos têm exatamente as mesmas 30 colunas.")


Colunas por ano: 2023: 30 | 2024: 30 | 2025: 30
Comuns aos três anos: 30
Só em alguns anos    : nenhuma

OK — os três anos têm exatamente as mesmas 30 colunas.


## 1.3 `arquivo_origem` e a união em `acidentes_prf_historico`

**Pergunta:** é possível consolidar os três anos numa única base rastreável?

`arquivo_origem` guarda o ano do arquivo. Na Parte 3 será criada `ano_acidente`, derivada de
`data_inversa`; as duas existindo, dá para checar se algum registro está no arquivo errado.


In [4]:
for ano, df in dados_por_ano.items():
    df["arquivo_origem"] = ano

acidentes_prf_historico = pd.concat(
    [dados_por_ano[ano] for ano in ANOS], ignore_index=True, sort=False
)

print(f"Formato de acidentes_prf_historico: {acidentes_prf_historico.shape}")
print("\nLinhas por arquivo de origem:")
print(acidentes_prf_historico["arquivo_origem"].value_counts().sort_index().to_string())
acidentes_prf_historico.head(3)


Formato de acidentes_prf_historico: (213497, 31)

Linhas por arquivo de origem:
arquivo_origem
2023    67766
2024    73202
2025    72529


,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop,arquivo_origem
0,496519,2023-01-01,domingo,02:00:00,ES,101,114,SOORETAMA,Ausência de reação do condutor,Saída de leito carroçável,Com Vítimas Feridas,Plena Noite,Crescente,Céu Claro,Simples,Reta,Não,1,0,1,0,0,0,1,1,"-19,09484877","-40,05095848",SPRF-ES,DEL04-ES,UOP01-DEL04-ES,2023
1,496543,2023-01-01,domingo,03:40:00,SP,116,"113,1",TAUBATE,Entrada inopinada do pedestre,Atropelamento de Pedestre,Com Vítimas Fatais,Plena Noite,Decrescente,Céu Claro,Dupla,Reta,Sim,5,1,0,0,0,4,0,2,"-23,0445658","-45,58259814",SPRF-SP,DEL02-SP,UOP02-DEL02-SP,2023
2,496590,2023-01-01,domingo,01:40:00,MT,163,1112,GUARANTA DO NORTE,Reação tardia ou ineficiente do condutor,Tombamento,NaN,Plena Noite,Crescente,Ignorado,Simples,Curva;Declive,Não,2,0,0,1,0,2,1,3,"-9,70020602","-54,87588757",SPRF-MT,DEL06-MT,UOP03-DEL06-MT,2023


## 1.4 Prova de que nenhuma linha foi perdida

**Pergunta:** o total consolidado é exatamente a soma dos três arquivos?

O `assert` precisa ser verificado **agora**, antes da remoção de duplicatas da Parte 2 —
medido depois, o teste falharia por construção.


In [5]:
total_esperado = sum(linhas_por_arquivo.values())
total_obtido = len(acidentes_prf_historico)

print("Soma dos arquivos :", f"{total_esperado:,}")
print("Base consolidada  :", f"{total_obtido:,}")

assert total_obtido == total_esperado, (
    f"Linhas perdidas na concatenação: {total_esperado - total_obtido}"
)
print("\nOK — nenhuma linha perdida na integração.")

pd.DataFrame({
    "linhas": linhas_por_arquivo,
    "fatais": {ano: int((df["mortos"] >= 1).sum()) for ano, df in dados_por_ano.items()},
})


Soma dos arquivos : 213,497
Base consolidada  : 213,497

OK — nenhuma linha perdida na integração.


,linhas,fatais
2023,67766,4858
2024,73202,5225
2025,72529,5210
